In [1]:
import os
import re
import sys
import contextlib
from pathlib import Path
from collections import defaultdict

# Numerical Computing & Data Handling:
import numpy as np
import pandas as pd

# Scientific Computing & Statistics:
from scipy.io import loadmat, savemat
from scipy.stats import ttest_ind
from statsmodels.stats.multitest import multipletests

# Neuroimaging:
import nibabel as nib
from nilearn.datasets import fetch_atlas_schaefer_2018
from nilearn.maskers import NiftiLabelsMasker
from neuromaps.datasets import fetch_fslr

# Network Analysis:
import networkx as nx
from networkx.algorithms import community

# Visualization:
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.colors as mcolors
import matplotlib.patches as patches
from matplotlib.lines import Line2D
import seaborn as sns

# Brain Surface Visualization:
from surfplot import Plot

# Utilities:
from tqdm.auto import tqdm

# Warning Suppression Utility:
@contextlib.contextmanager
def suppress_vtk_warnings():
    devnull = open(os.devnull, "w")
    old_stderr = os.dup(2)
    os.dup2(devnull.fileno(), 2)
    try:
        yield
    finally:
        os.dup2(old_stderr, 2)
        os.close(old_stderr)
        devnull.close()

In [2]:
activation_root = Path("outputs_yeo7_activation_all_methods")

# Chosen binary activation method
selected_suffix = "Z15"

# Fixed temporal window
window_size = 20

# Output folder
dataset_out = Path(f"model_dataset_{selected_suffix}_W{window_size}")
dataset_out.mkdir(exist_ok=True, parents=True)

network_names = [
    "Visual",
    "Somatomotor",
    "Dorsal_Attention",
    "Ventral_Attention",
    "Limbic",
    "Frontoparietal",
    "Default_Mode"
]

In [3]:
def create_windows_from_binary_file(csv_file, window_size=20):
    """
    Creates temporal windows from one binary activation CSV.

    X = previous window_size timepoints
    y = next timepoint
    """

    df = pd.read_csv(csv_file)

    # Make sure the networks are in the correct order
    values = df[network_names].values.astype(np.float32)

    X = []
    y = []

    for i in range(len(values) - window_size):
        X.append(values[i:i + window_size])
        y.append(values[i + window_size])

    X = np.array(X, dtype=np.float32)
    y = np.array(y, dtype=np.float32)

    return X, y

In [4]:
binary_files = sorted(
    activation_root.rglob(f"*_yeo7_binary_{selected_suffix}.csv")
)

print("Binary files found:", len(binary_files))

for f in binary_files[:5]:
    print(f)

Binary files found: 1086
outputs_yeo7_activation_all_methods/sub-OAS30001/ses-d0129/sub-OAS30001_ses-d0129_run-02_yeo7_binary_Z15.csv
outputs_yeo7_activation_all_methods/sub-OAS30002/ses-d0653/sub-OAS30002_ses-d0653_run-02_yeo7_binary_Z15.csv
outputs_yeo7_activation_all_methods/sub-OAS30003/ses-d0558/sub-OAS30003_ses-d0558_run-03_yeo7_binary_Z15.csv
outputs_yeo7_activation_all_methods/sub-OAS30004/ses-d1101/sub-OAS30004_ses-d1101_run-02_yeo7_binary_Z15.csv
outputs_yeo7_activation_all_methods/sub-OAS30005/ses-d0581/sub-OAS30005_ses-d0581_run-02_yeo7_binary_Z15.csv


In [5]:
X_all = []
y_all = []
metadata_rows = []

for csv_file in tqdm(binary_files):

    X, y = create_windows_from_binary_file(
        csv_file,
        window_size=window_size
    )

    if len(X) == 0:
        continue

    subject = next((p for p in csv_file.parts if p.startswith("sub-")), None)
    session = next((p for p in csv_file.parts if p.startswith("ses-")), None)

    run_parts = [p for p in csv_file.name.split("_") if p.startswith("run-")]
    run = run_parts[0] if len(run_parts) > 0 else "run-unknown"

    X_all.append(X)
    y_all.append(y)

    for i in range(len(X)):
        metadata_rows.append({
            "subject": subject,
            "session": session,
            "run": run,
            "source_file": str(csv_file),
            "window_start": i,
            "window_end": i + window_size - 1,
            "target_timepoint": i + window_size,
            "threshold_method": selected_suffix,
            "window_size": window_size
        })

X_all = np.concatenate(X_all, axis=0)
y_all = np.concatenate(y_all, axis=0)
sample_metadata = pd.DataFrame(metadata_rows)

print("X shape:", X_all.shape)
print("y shape:", y_all.shape)
print("Metadata shape:", sample_metadata.shape)

sample_metadata.head()

  0%|          | 0/1086 [00:00<?, ?it/s]

X shape: (115115, 20, 7)
y shape: (115115, 7)
Metadata shape: (115115, 9)


,subject,session,run,source_file,window_start,window_end,target_timepoint,threshold_method,window_size
0,sub-OAS30001,ses-d0129,run-02,outputs_yeo7_activation_all_methods/sub-OAS300...,0,19,20,Z15,20
1,sub-OAS30001,ses-d0129,run-02,outputs_yeo7_activation_all_methods/sub-OAS300...,1,20,21,Z15,20
2,sub-OAS30001,ses-d0129,run-02,outputs_yeo7_activation_all_methods/sub-OAS300...,2,21,22,Z15,20
3,sub-OAS30001,ses-d0129,run-02,outputs_yeo7_activation_all_methods/sub-OAS300...,3,22,23,Z15,20
4,sub-OAS30001,ses-d0129,run-02,outputs_yeo7_activation_all_methods/sub-OAS300...,4,23,24,Z15,20


In [6]:
print("Number of samples:", len(X_all))
print("Window size:", X_all.shape[1])
print("Number of networks:", X_all.shape[2])
print("Target shape:", y_all.shape)

Number of samples: 115115
Window size: 20
Number of networks: 7
Target shape: (115115, 7)


In [7]:
np.save(dataset_out / "X.npy", X_all)
np.save(dataset_out / "y.npy", y_all)
sample_metadata.to_csv(dataset_out / "sample_metadata.csv", index=False)

print("Saved dataset in:", dataset_out)

Saved dataset in: model_dataset_Z15_W20
